<a href="https://colab.research.google.com/github/Amruth-U-tech/DL-Journey/blob/main/05-LSTMs/SatckedLSTM-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance

In [2]:
import yfinance as yf
import numpy as np
import pandas as pd #for data handling or to read
import matplotlib.pyplot as plt #to print any 2D instance
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM,Add, Bidirectional, Input  #we got two RNNs here
from tensorflow.keras.optimizers import Adam

In [3]:
df = yf.download("AAPL",period="5y",auto_adjust=True)[["Close"]]  #we are dowloding "AAPL" which mean aplle stock market data of period 5years

#with the target variable as close which means there is a feature in that data known as close which shos the closing price of the stock and that Y
df.head(10)

[*********************100%***********************]  1 of 1 completed


Price,Close
Ticker,AAPL
Date,
2021-03-17,121.515686
2021-03-18,117.395699
2021-03-19,116.869728
2021-03-22,120.181313
2021-03-23,119.353432
2021-03-24,116.967110
2021-03-25,117.454109
2021-03-26,118.057999


In [4]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)
scaled_data

array([[2.79862480e-02],
       [3.62785615e-03],
       [5.18187838e-04],
       ...,
       [7.88326944e-01],
       [8.04290088e-01],
       [8.09197176e-01]])

In [5]:
from pandas.core.window.rolling import Window
#now because this is Sequential data we make our x and Y in such a way itself
#such that some input X1 will give some input Y1 then that Y1 is included with X1 by removing previous data of X1 hence X2=X1-X1[0]+Y1 we get Y2

def create_sequence(data,window=30):
  X,Y = [],[]
  for i in range(window,len(data)):
    X.append(data[i-window:i])
    Y.append(data[i])
  return np.array(X),np.array(Y)

Window_size = 30
x,y = create_sequence(scaled_data,window=Window_size)

In [6]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,shuffle=False)
print(x_train.shape)
print(x_test.shape)

(980, 30, 1)
(246, 30, 1)


In [7]:
def complie_and_train(model,name,epochs=10,batch_size=32):
    model.compile(optimizer='adam',loss='mse')
    history = model.fit(x_train,y_train,epochs=epochs,batch_size=batch_size,verbose=1)

    print(f"{name} trained")
    return history

In [8]:
Window_size=30
batch_size = 32
epochs =10
model_stackedLSTM = Sequential([Input(shape=(Window_size,1)),LSTM(50,return_sequences=True),LSTM(50),Dense(1)])
model_stackedLSTM.summary()

hist_stackedLSTM = complie_and_train(model_stackedLSTM,"stackedLSTM",epochs=epochs,batch_size=batch_size)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 50)         │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,651 (119.73 KB)

 Trainable params: 30,651 (119.73 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 0.0273
Epoch 2/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0025
Epoch 3/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0017
Epoch 4/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0017
Epoch 5/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0015
Epoch 6/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0015
Epoch 7/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0014
Epoch 8/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0014
Epoch 9/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0013
Epoch 10/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0012
stackedLSTM trained


In [9]:
#model prediction
prediction = model_stackedLSTM.predict(x_test)
prediction=scaler.inverse_transform(prediction)
y_test1 = scaler.inverse_transform(y_test)

for i in range(len(prediction)):
  print(f"prdeiction: {prediction[i][0]:.2f}, actual:{y_test1[i][0]:.2f}")

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 96ms/step
prdeiction: 216.85, actual:222.78
prdeiction: 216.82, actual:220.57
prdeiction: 217.12, actual:222.88
prdeiction: 217.78, actual:216.95
prdeiction: 218.29, actual:221.17
prdeiction: 218.92, actual:222.22
prdeiction: 219.65, actual:222.92
prdeiction: 220.44, actual:202.31
prdeiction: 220.01, actual:187.56
prdeiction: 217.93, actual:180.67
prdeiction: 214.43, actual:171.67
prdeiction: 209.63, actual:197.99
prdeiction: 205.74, actual:189.59
prdeiction: 202.15, actual:197.29
prdeiction: 199.49, actual:201.64
prdeiction: 197.86, actual:201.26
prdeiction: 197.02, actual:193.43
prdeiction: 196.33, actual:196.13
prdeiction: 195.95, actual:192.32
prdeiction: 195.57, actual:198.87
prdeiction: 195.57, actual:203.71
prdeiction: 196.10, actual:207.47
prdeiction: 197.19, actual:208.37
prdeiction: 198.66, actual:209.23
prdeiction: 200.37, actual:210.29
prdeiction: 202.22, actual:211.58
prdeiction: 204.13, actual:212.39
prdeiction: 206.03, actual:204.46
prdeiction